# Gemma 4 — fine-tune + evaluate (Colab) → adapter for local RTX 3090 inference

**Policy:** Colab does **fine-tuning + evaluation only**; final/production inference runs on the local RTX 3090.
The produced LoRA adapter is small (~0.5 GB) and serves on the 3090 (Gemma 4 12B is 6.7 GB at 4-bit).

**Model menu** (largest that fits a 24 GB 3090 for inference at 4-bit = **12B**):

| model | HF id | 4-bit (inference) |
|---|---|---|
| Gemma 4 12B (recommended, max for 24GB) | `unsloth/gemma-4-12b-it` | 6.7 GB |
| Gemma 4 E4B (faster) | `unsloth/gemma-4-E4B-it` | 4.5 GB |
| Gemma 4 E2B (fastest) | `google/gemma-4-E2B-it` | 2.9 GB |

**Drive flow:** code+data come zipped from the shared folder
`MyDrive/ehrsql/EHRSQL_GEMMA/ehrsql_gemma_bundle.zip` (a shortcut to your 5 TB
account's shared folder) → copied to the **local Colab disk**. Adapters + eval
results are written **back** to that folder. Add a Colab **Secret `HF_TOKEN`** first.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Copy + unzip the bundle from the shared folder to local disk


In [ ]:
import os
# The 'ehrsql' folder in the Colab account's Drive holds a shortcut to the shared
# folder 'EHRSQL_GEMMA' (owned by your 5TB account).
DRIVE_DIR = '/content/drive/MyDrive/ehrsql/EHRSQL_GEMMA'
BUNDLE = f'{DRIVE_DIR}/ehrsql_gemma_bundle.zip'
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} (upload the zip + share the folder first)'
os.makedirs('/content/ehrcopilot', exist_ok=True)
!unzip -q -o "$BUNDLE" -d /content/ehrcopilot
os.chdir('/content/ehrcopilot')
print('cwd:', os.getcwd()); !ls

## 3. Install dependencies (Gemma 4 = transformers 5.10.1 + Unsloth-from-git)

> **If you already ran the old install** (transformers 5.5.0 in this session): after this
> cell, do **Runtime → Restart session**, then run from the *next* cell (skip re-installing).
> A fresh runtime needs no restart.


In [ ]:
# Gemma 4 (gemma4_unified) needs Unsloth-from-git, which pulls its OWN compatible
# transformers/trl/peft. Do NOT pin transformers — pinning it conflicts with the
# git unsloth_zoo (it currently wants transformers 5.12.x).
!pip -q install "unsloth @ git+https://github.com/unslothai/unsloth" \
               "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo"
# Orthogonal project deps (do not touch the transformers/peft/trl stack):
!pip -q install datasets sentencepiece timm scikit-learn rank-bm25 sqlglot faiss-cpu einops
import transformers, unsloth; print('transformers', transformers.__version__, '| unsloth', unsloth.__version__)


## 4. HF token (Colab Secret) + environment + pick model + scale batch to GPU


In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')          # gated Gemma 4 access
os.environ['PYTHONPATH'] = 'src'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# >>> choose the model (12B = max that runs inference on the 24GB 3090) <<<
MODEL = 'unsloth/gemma-4-12b-it'   # TEXT model, FastModel path, max that fits a 24GB 3090 at 4-bit
# NOTE: E2B/E4B are multimodal (need FastVisionModel) — NOT this text pipeline.
# 26B-A4B / 31B are text (FastModel) but too big for 24GB inference. Keep 12B.
SFT_EPOCHS = 3                        # the lever for EX (1 epoch plateaued ~0.40)
ORPO_EPOCHS = 2

import torch
p = torch.cuda.get_device_properties(0); VRAM = p.total_memory/1e9
print(f'GPU: {p.name}  {VRAM:.0f} GB   MODEL={MODEL}')
BS, GA = (16,1) if VRAM>70 else (8,2) if VRAM>38 else (2,8) if VRAM>20 else (1,16)
print('batch_size', BS, 'grad_accum', GA, '(effective', BS*GA, ')')

## 5. Prepare data (build DB / SFT data if not in the bundle)


In [ ]:
import os
if not os.path.exists('data/mimic_iv_demo.db'):
    print('Building MIMIC-IV-Demo SQLite (open access)...')
    !wget -q -r -N -np -P /content/mimicdl https://physionet.org/files/mimic-iv-demo/2.2/
    !mkdir -p data/mimic-iv-demo && cp -r /content/mimicdl/physionet.org/files/mimic-iv-demo/2.2/. data/mimic-iv-demo/
    !PYTHONPATH=src python -m ehrcopilot.db.build_sqlite data/mimic-iv-demo data/mimic_iv_demo.db
if not os.path.exists('data/ehrsql/sft_train_v2.jsonl'):
    !PYTHONPATH=src python -m ehrcopilot.finetune.prepare_sft \
      --train data/ehrsql/ehrsql/mimic_iii/train.json --valid data/ehrsql/ehrsql/mimic_iii/valid.json \
      --output data/ehrsql/sft_train_v2.jsonl
!ls -la data/mimic_iv_demo.db data/ehrsql/sft_train_v2.jsonl

## 6. QLoRA SFT (Gemma 4)


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.qlora_sft \
  --base-model {MODEL} --data data/ehrsql/sft_train_v2.jsonl --output checkpoints/sft_gemma4 \
  --epochs {SFT_EPOCHS} --batch-size {BS} --grad-accum {GA}

## 7. Build Abstention-ORPO pairs


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.build_pairs \
  --train data/ehrsql/ehrsql/mimic_iii/train.json --valid data/ehrsql/ehrsql/mimic_iii/valid.json \
  --adapter checkpoints/sft_gemma4/adapter_final --output data/ehrsql/gemma4_orpo_pairs.jsonl \
  --max-answerable 500 --verify-execution

## 8. Abstention-ORPO (calibrates [ABSTAIN] → positive RS)


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.finetune.abstention_dpo \
  --pairs data/ehrsql/gemma4_orpo_pairs.jsonl --adapter checkpoints/sft_gemma4/adapter_final \
  --output checkpoints/orpo_gemma4 --epochs {ORPO_EPOCHS} --max-length 1536

## 9. Evaluate (full 1786 test set) — classifier retriever + repair


In [ ]:
!PYTHONPATH=src python -m ehrcopilot.eval.harness data/ehrsql/ehrsql/mimic_iii/test.json \
  --model checkpoints/orpo_gemma4/adapter_final --few-shot data/ehrsql/ehrsql/mimic_iii/train.json \
  --retrieval-mode classifier --repair --output tests/evalgen/gemma4_orpo_full.json
import json; print(json.load(open('tests/evalgen/gemma4_orpo_full.json')))

## 10. Save adapters + results back to Drive (for local 3090 inference)


In [ ]:
DEST = f'{DRIVE_DIR}/checkpoints'
!mkdir -p "$DEST"
!cp -r checkpoints/sft_gemma4  "$DEST"/
!cp -r checkpoints/orpo_gemma4 "$DEST"/
!mkdir -p "$DRIVE_DIR"/results && cp tests/evalgen/gemma4_orpo_full.json "$DRIVE_DIR"/results/ 2>/dev/null || true
print('Saved adapters + results to', DEST)
print('Download checkpoints/orpo_gemma4/adapter_final → run inference locally on the RTX 3090.')

## Local inference (RTX 3090)
```bash
# after downloading orpo_gemma4/adapter_final into checkpoints/orpo_gemma4/
PYTHONPATH=src python -m ehrcopilot.eval.harness data/ehrsql/ehrsql/mimic_iii/test.json \
  --model checkpoints/orpo_gemma4/adapter_final \
  --few-shot data/ehrsql/ehrsql/mimic_iii/train.json --retrieval-mode classifier --repair
```
